In [0]:
import pandas as pd 
from matplotlib import pyplot as plt 
import seaborn as sns 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import FunctionTransformer
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from mlflow.models import infer_signature
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
import numpy as np
import json
import os
import requests
from pyspark.sql.functions import col, when

In [0]:
BASE_DIR = "/Volumes/workspace/default/data_customers/raw/"
train_path = f"{BASE_DIR}customer_churn_dataset-training-master.csv"
df_train = spark.read.option("header", "true").option("inferSchema", "true").csv(train_path)
df_train = df_train.dropna()
df_train = df_train.dropDuplicates()
df_train = df_train.withColumnRenamed("customerid", "customer_id")
df_train = df_train.withColumn("customer_id", col("customer_id").cast("string"))
display(df_train)

In [0]:
MODEL_REGISTRY_NAME = "workspace.default.customer_churn_model"
FEATURE_TABLE_NAME = "workspace.default.customer_churn_features"

fe = FeatureEngineeringClient()
label_df = df_train.select("customer_id", "churn")

feature_lookups = [
    FeatureLookup(
        table_name=FEATURE_TABLE_NAME,
        lookup_key="customer_id"
    )
]

training_set = fe.create_training_set(
    df=label_df,
    feature_lookups=feature_lookups,
    label="churn"
)
df = training_set.load_df().toPandas()
df.head()

In [0]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["churn"]
)

X_train = df_train.drop(columns=["churn"])
y_train = df_train["churn"]

X_test = df_test.drop(columns=["churn"])
y_test =  df_test["churn"]

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [0]:
raw_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=0.1,
        penalty="l2",
        class_weight="balanced",
        random_state=42,
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
        eval_metric="logloss",
    ),
}

results = {}

for name, model in raw_models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ]
    )

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=["accuracy", "recall", "roc_auc"],
    )

    results[name] = {
        "AUC": cv_results["test_roc_auc"].mean(),
        "Recall": cv_results["test_recall"].mean(),
        "Accuracy": cv_results["test_accuracy"].mean(),
    }

results_df = pd.DataFrame(results).T.sort_values(by="AUC", ascending=False)
best_model_name = results_df.index[0]

print("Best model:", best_model_name)
display(results_df)

In [0]:
final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", raw_models[best_model_name]),
    ]
)

final_pipeline.fit(X_train, y_train)

y_test_pred = final_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred))

signature = infer_signature(X_train, final_pipeline.predict(X_train))

In [0]:
user = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user}/churn_experiment")

with mlflow.start_run(run_name=best_model_name, nested=True):
    mlflow.log_param("model_name", best_model_name)

    mlflow.log_metric("accuracy", results_df.loc[best_model_name, "Accuracy"])
    mlflow.log_metric("auc", results_df.loc[best_model_name, "AUC"])
    mlflow.log_metric("recall", results_df.loc[best_model_name, "Recall"])

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="model",
        signature=signature,
        registered_model_name=MODEL_REGISTRY_NAME,
    )

print("Model logged to MLflow!")

In [0]:
client = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'")

# Lấy version lớn nhất để tránh nhầm thứ tự trả về từ API
latest_version = max(int(v.version) for v in versions)
client.set_registered_model_alias(MODEL_REGISTRY_NAME, "champion", str(latest_version))

print(f"Đã gắn alias @champion cho Version {latest_version} thành công!")




In [0]:
workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl") 
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

model_name = "workspace.default.customer_churn_model"
endpoint_name = "customer-churn-endpoint" 

versions = client.search_model_versions(f"name='{model_name}'")
latest_version = versions[0].version
print(f"🔄 Đang tiến hành cập nhật Endpoint lên Version mới nhất: {latest_version}")

update_payload = {
  "served_entities": [
    {
      "name": "current_model",
      "entity_name": model_name,
      "entity_version": str(latest_version), 
      "scale_to_zero_enabled": True,
      "workload_size": "Small"
    }
  ]
}

headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
endpoint_url = f"{workspace_url}/api/2.0/serving-endpoints/{endpoint_name}/config"
response = requests.put(endpoint_url, headers=headers, data=json.dumps(update_payload))